#  Arm Assembly Primer : Arguments and return values
# 1. Registers
* A CPU has a small number of <font color="blue">registers</font>, storage locations inside the CPU, each holding one value
* Instructions mostly compute on registers (e.g., `add w0, w0, #1`); values in memory are first brought into registers
* When you write a C/C++ program, the compiler decides which register holds which variable. Roughly speaking, <font color="blue">local variables and parameters of a function become registers</font> (unless there are too many of them, or their addresses are taken)
* Arm (AArch64) has these registers for ordinary computation

<div>

| name | size | used for |
|------|------|----------|
| `x0` -- `x30` | 64 bits | integers and pointers (`long`, `int *`, ...) |
| `w0` -- `w30` | 32 bits | the lower half of `x0` -- `x30` (`int`) |
| `d0` -- `d31` | 64 bits | floating point numbers (`double`) |
| `s0` -- `s31` | 32 bits | the lower half of `d0` -- `d31` (`float`) |
| `sp` | 64 bits | the stack pointer (previous notebook) |
| `xzr` / `wzr` | 64 / 32 bits | always zero |

</div>

* `w0` is not a separate register; it is the lower 32 bits of `x0`. Writing to `w0` sets the upper 32 bits of `x0` to zero
* Likewise, `s0` is the lower 32 bits of `d0`

# 2. Where are the parameters? (the called function's view)
* The rule (_calling convention_) decides which registers hold the arguments and the return value, so that separately compiled functions can call each other


In [ ]:
%%writefile args.c
long f_long(long a, long b, long c) {
  return a - b + c + 123;
}

int f_int(int a, int b) {
  return a - b + 45;
}

double f_double(double x, double y) {
  return x - y + 1.5;
}

float f_float(float x, float y) {
  return x - y + 2.5f;
}

double f_mixed(int n, double x, long m, float y) {
  return n + x + m + y + 0.25;
}

long * f_ptr(long * p, long * q) {
  return q + 3;
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S args.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' args.s

* You will see something like this

```
f_long:
	sub	x8, x0, x1           // x8 = a - b
	add	x8, x8, x2           // x8 = x8 + c
	add	x0, x8, #123         // return x8 + 123
	ret
f_int:
	sub	w8, w0, w1           // w8 = a - b
	add	w0, w8, #45          // return w8 + 45
	ret
f_double:
	fsub	d0, d0, d1           // d0 = x - y
	fmov	d1, #1.50000000      // d1 = 1.5
	fadd	d0, d0, d1           // return d0 + 1.5
	ret
f_float:
	fsub	s0, s0, s1           // s0 = x - y
	fmov	s1, #2.50000000      // s1 = 2.5
	fadd	s0, s0, s1           // return s0 + 2.5
	ret
f_mixed:
	scvtf	d3, w0               // d3 = (double)n
	scvtf	d2, x1               // d2 = (double)m
	fcvt	d1, s1               // d1 = (double)y
	fadd	d0, d0, d3           // d0 = x + n
	fadd	d0, d0, d2           //    + m
	fadd	d0, d0, d1           //    + y
	fmov	d1, #0.25000000
	fadd	d0, d0, d1           //    + 0.25
	ret
f_ptr:
	add	x0, x1, #24          // return q + 3 (3 longs = 24 bytes ahead)
	ret
```

* Reading them backwards from the constants (`#123`, `#45`, `#1.5`, ...), you can tell which register holds which parameter
  * `f_long` : `a`, `b`, `c` are in `x0`, `x1`, `x2`, and the result goes to `x0`
  * `f_int` : `a`, `b` are in `w0`, `w1` (32-bit `int`, so the `w` names), and the result goes to `w0`
  * `f_double` : `x`, `y` are in `d0`, `d1`, and the result goes to `d0` (`fsub` and `fadd` are the floating point subtraction and addition)
  * `f_float` : `x`, `y` are in `s0`, `s1`, and the result goes to `s0`
  * `f_mixed` : `n` in `w0`, `x` in `d0`, `m` in `x1`, `y` in `s1`; integer registers and floating point registers are numbered _separately_
  * `f_ptr` : pointers are in `x0`, `x1`, just like `long`; the result is returned in `x0`. Note that `q + 3` adds 24, not 3: `q` points to `long`s (8 bytes each), and `q + 3` is the address of the third `long` after the one `q` points to
* `x8` and `w8` hold intermediate results; the compiler picks such a register for temporary values (`x8` -- `x15` are typical choices; they are not used for arguments)
* Floating point constants are first put into a register with `fmov`, as floating point arithmetic instructions cannot take a constant operand
* In summary

<div>

| | arguments | return value |
|-|-----------|--------------|
| integers, pointers | `x0`, `x1`, ..., `x7` (`w0`, ... for 32-bit values) | `x0` (`w0`) |
| floating point numbers | `d0`, `d1`, ..., `d7` (`s0`, ... for `float`) | `d0` (`s0`) |

</div>

* Arguments beyond eight are passed on the stack (in memory)
* That pointers are passed in the same registers as `long` is not a coincidence: <font color="blue">a pointer is a 64-bit integer, the address of the data it points to</font> (more on this in pd09_access)
* `f_mixed` also shows conversions: `scvtf` converts an integer (`n`, `m`) into a `double`, and `fcvt d1, s1` converts the `float` `y` into a `double` (next notebook)

# 3. Where to put the arguments? (the calling function's view)
* The calling function puts the arguments into these registers before `bl`, and finds the result in `x0` (or `d0`, ...) afterwards


In [ ]:
%%writefile caller.c
long f_long(long a, long b, long c);                 /* defined elsewhere */
double f_mixed(int n, double x, long m, float y);    /* defined elsewhere */

long call_long(void) {
  return f_long(10, 20, 30) + 1;
}

double call_mixed(void) {
  return f_mixed(1, 2.0, 3, 4.0f);
}

In [ ]:
%%bash
module purge && module load nvidia/25.9 cuda/12.9 && PATH=/work/gt81/share/env/llvm/bin:$PATH
clang -O1 -S caller.c
grep -vE '^\s+\.|^\s*//|^\.Lfunc' caller.s

* You will see something like this

```
call_long:
	stp	x29, x30, [sp, #-16]!
	mov	x29, sp
	mov	w0, #10           // 1st argument
	mov	w1, #20           // 2nd argument
	mov	w2, #30           // 3rd argument
	bl	f_long
	add	x0, x0, #1        // the result of f_long is in x0
	ldp	x29, x30, [sp], #16
	ret
call_mixed:
	fmov	d0, #2.00000000   // x (the 1st floating point argument)
	fmov	s1, #4.00000000   // y (the 2nd floating point argument)
	mov	w0, #1            // n (the 1st integer argument)
	mov	w1, #3            // m (the 2nd integer argument)
	b	f_mixed
```

* `mov w0, #10` sets the `long` argument `a` to 10: writing to `w0` clears the upper half of `x0`, so `x0` becomes 10 (and the instruction is a bit shorter to encode than `mov x0, #10`)
* `add x0, x0, #1` uses the return value of `f_long`, found in `x0`, and leaves its own return value in `x0`
* In `call_mixed`, the call is the last thing to do, so it is a tail call (`b f_mixed`), and the return value of `f_mixed` (in `d0`) becomes that of `call_mixed` as it is

# 4. Summary
* parameters and local variables mostly live in registers
* integer and pointer arguments: `x0` -- `x7` (`w0` -- `w7` for 32-bit values); floating point arguments: `d0` -- `d7` (`s0` -- `s7` for `float`); they are numbered separately
* the return value is in `x0` / `w0` / `d0` / `s0`
* a pointer is passed just like a `long`: it is a 64-bit integer (an address)